## Transform Circuits Data

1. Read bronze_circuits table
2. Keep only the columns required for analytics (Drop url column)
3. Standardise column names using snake_case (circuitId → circuit_id, circuitName → circuit_name)
4. Rename columns to make them more meaningful (lat → latitude, long → longitude)
5. Filter out rows where circuit_id is null (business key validation)
6. Remove duplicate records
7. Transform values of columns circuit_name and locality to Title Case
8. Write the transformed data to silver_circuits table


## Below changes are required to implement Incremental Load Processing

1. Accept batch_id as a parameter to the notebook
2. Process data for only the `batch_id` being passed in (i.e., filter reading from bronze using the `batch_id`)
3. Add `created_timestamp`, `updated_timestamp` and `batch_id` to the silver table.
4. Merge the processed data to the silver table
    * `created_timestamp` should only be populated at the time of inserting/creating the record. It should not be updated during the merge.
    * Ensure that we are not overwriting the data in the silver table by older bronze data (re-run scenario)

### Step 1- Read bronze_circuits table

In [0]:
dbutils.widgets.text("p_batch_id","")
v_batch_id = dbutils.widgets.get("p_batch_id")

In [0]:
%run ../00-common/01.environment-configuration

In [0]:
%run ../00-common/03.Silver-Helper

In [0]:
bronze_table = f"{catalog_name}.{bronze_schema}.circuits"
silver_table = f"{catalog_name}.{silver_schema}.circuits"


### Step 1- Read bronze_circuits table

In [0]:
from pyspark.sql import functions as F

In [0]:
circuit_df = (
    spark.read.table(bronze_table).filter((F.col("batch_id") == v_batch_id))
)

  

In [0]:
display(circuit_df)


#### Keep only the columns required for analytics (Drop url column)

In [0]:
circuit_selected_df = circuit_df.select(
    F.col("circuitId"),
    F.col("circuitName"),
    F.col("lat"),
    F.col("long"),
    F.col("locality"),
    F.col("country"),
    F.col("timestamp"),
    F.col("source_file"),
    F.col("batch_id"),
)

In [0]:
display(circuit_selected_df)


#### Standardise column names using snake_case (circuitId → circuit_id, circuitName → circuit_name)
#### Rename columns to make them more meaningful (lat → latitude, long → longitude)

In [0]:
circuits_remaned_df = (
    circuit_selected_df
        .withColumnRenamed("circuitId", "circuit_id")
        .withColumnRenamed("circuitName", "circuit_name")                
        .withColumnRenamed("lat", "latitude")
        .withColumnRenamed("long", "longitude")
)

In [0]:
display(circuits_remaned_df)


In [0]:
circuits_remaned_df = (
    circuit_selected_df
        .withColumnsRenamed({
            "circuitId": "circuit_id",
            "circuitName": "circuit_name",
            "lat": "latitude",
            "long": "longitude"
        })
)

- Filter out rows where circuit_id is null (business key validation)


In [0]:
# SQL Expression 
# circuits_valid_df = (
#    circuits_remaned_df
#        .filter("circuit_id is not null")
#)

# Column Expression
circuits_valid_df = (
    circuits_remaned_df
       .filter(F.col("circuit_id").isNotNull()          
        )
)


In [0]:
display(circuits_valid_df)

- Remove duplicate records

In [0]:
circuits_distinct_df = circuits_valid_df.distinct()


In [0]:
circuits_distinct_df = circuits_valid_df.dropDuplicates(["circuit_id"])

In [0]:
display(circuits_distinct_df)


### Transform values of columns circuit_name and locality to Title Case

In [0]:
circuits_final_df = (
    circuits_distinct_df
        .withColumn("circuit_name", F.initcap(F.col("circuit_name")))
        .withColumn("locality", F.initcap(F.col("locality")))        
)


In [0]:
display(circuits_final_df)


### Write a data to silver_table

In [0]:


# circuits_final_df = (
#    circuits_final_df
#        .withColumn("created_timestamp", F.current_timestamp())
#        .withColumn("updated_timestamp", F.current_timestamp())
# )

In [0]:
write_to_silver(
    input_df = circuits_final_df,
    target_table = silver_table,
    merge_condition= "t.circuit_id = s.circuit_id",
    columns_to_update=[
        "circuit_name",
        "latitude",
        "longitude",
        "country",
        "locality",
        "timestamp",
        "source_file",
        "batch_id"
    ]
)

if not spark.catalog.tableExists(silver_table):
    (
        circuits_final_df
            .write
            .mode("overwrite")
            .format("delta")
            .saveAsTable(silver_table)
    )
else:
    from delta.tables import DeltaTable

    delta_table = DeltaTable.forName(spark, silver_table)
    (
        delta_table.alias("t")
            .merge(
                circuits_final_df.alias("s"),
                "t.circuit_id = s.circuit_id"
            )
        .whenMatchedUpdate(
                condition = "s.batch_id >= t.batch_id",
                set = {
                    "circuit_name": "s.circuit_name",
                    "locality": "s.locality",
                    "country": "s.country",
                    "latitude": "s.latitude",
                    "longitude": "s.longitude",
                    "timestamp": "s.timestamp",
                    "batch_id": "s.batch_id",
                    "source_file": "s.source_file",
                    "updated_timestamp": "s.updated_timestamp"
                }
            )
        .whenNotMatchedInsertAll()
        .execute()
    )


In [0]:
display(spark.table(silver_table))